In [28]:
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [29]:
data_path = "../data/The Hague/Source A/Stationsbuurt_2022-04-27.json"

In [30]:
with open(data_path, "r") as f:
    cj = json.load(f)

In [31]:
vertices = np.array(cj["vertices"], dtype=float)
if "transform" in cj:
    scale = np.array(cj["transform"]["scale"])
    translate = np.array(cj["transform"]["translate"])
    vertices = vertices * scale + translate

In [32]:
def convert_to_lod1(cj):
    new_vertices = cj["vertices"].copy()

    for obj_id, city_obj in cj.get("CityObjects", {}).items():
        new_geometries = []

        for geom in city_obj.get("geometry", []):
            if geom["type"] != "Solid" or "semantics" not in geom:
                new_geometries.append(geom)
                continue

            semantics = geom["semantics"]
            surfaces = semantics.get("surfaces", [])
            values = semantics.get("values", [])

            ground_idx, roof_idx, wall_idx = -1, -1, -1

            for i, s in enumerate(surfaces):
                if s["type"] == "GroundSurface": ground_idx = i
                if s["type"] == "RoofSurface": roof_idx = i
                if s["type"] == "WallSurface": wall_idx = i

            if ground_idx == -1 or roof_idx == -1:
                new_geometries.append(geom)
                continue

            new_surfaces = surfaces.copy()
            if wall_idx == -1:
                wall_idx = len(new_surfaces)
                new_surfaces.append({"type": "WallSurface"})

            ground_rings = []
            max_z_int = -float('inf')

            for shell_i, shell in enumerate(geom.get("boundaries", [])):
                for surf_i, surface in enumerate(shell):
                    if not surface: continue
                    sem_val = values[shell_i][surf_i]

                    if sem_val == ground_idx:
                        ground_rings.append(surface[0])
                    elif sem_val == roof_idx:
                        for ring in surface:
                            for v_idx in ring:
                                max_z_int = max(max_z_int, cj["vertices"][v_idx][2])

            if not ground_rings or max_z_int == -float('inf'):
                new_geometries.append(geom)
                continue

            new_shell = []
            new_sem_values = []

            for ring in ground_rings:
                new_shell.append([ring])
                new_sem_values.append(ground_idx)

                roof_ring = []
                for v_idx in ring:
                    v = cj["vertices"][v_idx]
                    new_v = [v[0], v[1], max_z_int]
                    new_v_idx = len(new_vertices)
                    new_vertices.append(new_v)
                    roof_ring.append(new_v_idx)

                new_shell.append([roof_ring[::-1]])
                new_sem_values.append(roof_idx)

                n = len(ring)
                for i in range(n):
                    v_curr = ring[i]
                    v_next = ring[(i + 1) % n]
                    r_curr = roof_ring[i]
                    r_next = roof_ring[(i + 1) % n]

                    wall_face = [v_next, v_curr, r_curr, r_next]

                    new_shell.append([wall_face])
                    new_sem_values.append(wall_idx)

            new_geom = {
                "type": "Solid",
                "lod": "1",
                "boundaries": [new_shell],
                "semantics": {
                    "surfaces": new_surfaces,
                    "values": [[new_sem_values]]
                }
            }
            new_geometries.append(new_geom)

        city_obj["geometry"] = new_geometries

    cj["vertices"] = new_vertices

    if "metadata" not in cj:
        cj["metadata"] = {}
    cj["metadata"]["datasetLod"] = "1"

    return cj

In [33]:
import plotly.graph_objects as go
from plotly.colors import qualitative

def visualize_cityjson_3d(city_objects, vertices):
    fig = go.Figure()
    palette = qualitative.Alphabet

    for i, obj in enumerate(city_objects):
        obj_color = palette[i % len(palette)]
        obj_name = obj.get("type", f"Object_{i}")
        show_in_legend = True

        for geom in obj.get("geometry", []):
            boundaries = geom.get("boundaries", [])

            if geom["type"] == "Solid":
                boundaries = [surface for shell in boundaries for surface in shell]
            elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
                continue

            for surface in boundaries:
                outer_ring = surface[0]
                poly_pts = vertices[outer_ring]

                x = np.append(poly_pts[:, 0], poly_pts[0, 0])
                y = np.append(poly_pts[:, 1], poly_pts[0, 1])
                z = np.append(poly_pts[:, 2], poly_pts[0, 2])

                fig.add_trace(go.Scatter3d(
                    x=x, y=y, z=z,
                    mode='lines',
                    line=dict(color=obj_color, width=3),
                    name=obj_name,
                    legendgroup=str(i),
                    showlegend=show_in_legend
                ))
                show_in_legend = False

    fig.update_layout(
        title="Visualized CityObjects",
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(itemsizing='constant')
    )

    fig.show()

In [35]:
cto = [cj["CityObjects"]['bag_0518100000229788']] # 'bag_0518100000286900'
visualize_cityjson_3d(cto, vertices)

In [36]:
cjlod1 = convert_to_lod1(cj)
vertices_lod1 = np.array(cjlod1["vertices"], dtype=float)
if "transform" in cjlod1:
    scale = np.array(cjlod1["transform"]["scale"])
    translate = np.array(cjlod1["transform"]["translate"])
    vertices_lod1 = vertices_lod1 * scale + translate

In [37]:
cto1 = [cjlod1["CityObjects"]['bag_0518100000229788']]
visualize_cityjson_3d(cto1, vertices_lod1)

In [20]:
lod1 = "../data/The Hague/temp.json"
with open(lod1, "r") as f:
    cjlod1 = json.load(f)

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go

target_object_id = list(cj.get("CityObjects", {}).keys())[0]
city_obj = cj["CityObjects"][target_object_id]

fig = go.Figure()

for geom in city_obj.get("geometry", []):
    boundaries = geom.get("boundaries", [])

    if geom["type"] == "Solid":
        boundaries = [surface for shell in boundaries for surface in shell]
    elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
        continue

    for surface in boundaries:
        outer_ring = surface[0]
        poly_pts = vertices[outer_ring]

        # Appending the first vertex to the end is mathematically required 
        # to close the geometric loop of the line trace
        x = np.append(poly_pts[:, 0], poly_pts[0, 0])
        y = np.append(poly_pts[:, 1], poly_pts[0, 1])
        z = np.append(poly_pts[:, 2], poly_pts[0, 2])

        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode='lines',
            line=dict(color='#2c3e50', width=3),
            showlegend=False
        ))

fig.update_layout(
    title=f"Rotatable Wireframe: {target_object_id}",
    scene=dict(aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [7]:
import json
import numpy as np
import torch

def create_graph_dataset(filepath):
    with open(filepath, "r") as f:
        cj = json.load(f)

    v_raw = np.array(cj["vertices"], dtype=float)
    if "transform" in cj:
        v_raw = v_raw * cj["transform"]["scale"] + cj["transform"]["translate"]

    dataset = []

    for obj_id, city_obj in cj.get("CityObjects", {}).items():
        geom_list = city_obj.get("geometry", [])
        if not geom_list:
            continue

        edges = set()
        active_vertex_indices = set()

        for geom in geom_list:
            boundaries = geom.get("boundaries", [])
            if geom["type"] == "Solid":
                boundaries = [surface for shell in boundaries for surface in shell]
            elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
                continue

            for surface in boundaries:
                ring = surface[0]
                for i in range(len(ring)):
                    active_vertex_indices.add(ring[i])
                    u = ring[i]
                    v = ring[(i + 1) % len(ring)]
                    edges.add((u, v))
                    edges.add((v, u))

        if not active_vertex_indices:
            continue

        idx_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted(active_vertex_indices))}

        node_features = np.array([v_raw[old_idx] for old_idx in sorted(active_vertex_indices)])
        edge_index = np.array([[idx_map[u], idx_map[v]] for u, v in edges]).T

        x_tensor = torch.tensor(node_features, dtype=torch.float32)
        edge_index_tensor = torch.tensor(edge_index, dtype=torch.long)

        dataset.append({
            "id": obj_id,
            "x": x_tensor,
            "edge_index": edge_index_tensor,
            "type": city_obj.get("type", "Unknown")
        })

    return dataset

In [8]:
tensor_dataset = create_graph_dataset(data_path)

In [9]:
import json
import numpy as np
import torch

def create_sequence_dataset(filepath, sep_value=1e9):
    with open(filepath, "r") as f:
        cj = json.load(f)

    v_raw = np.array(cj["vertices"], dtype=float)
    if "transform" in cj:
        v_raw = v_raw * cj["transform"]["scale"] + cj["transform"]["translate"]

    v_raw = np.hstack((v_raw, np.zeros((v_raw.shape[0], 1))))
    sep_token = np.array([[0.0, 0.0, 0.0, 1.0]], dtype=float)

    dataset = []

    for obj_id, city_obj in cj.get("CityObjects", {}).items():
        geom_list = city_obj.get("geometry", [])
        if not geom_list:
            continue

        obj_sequence = []

        for geom in geom_list:
            boundaries = geom.get("boundaries", [])

            if geom["type"] == "Solid":
                boundaries = [surface for shell in boundaries for surface in shell]
            elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
                continue

            for surface in boundaries:
                ring = surface[0]
                ring_coords = v_raw[ring]
                obj_sequence.append(ring_coords)
                obj_sequence.append(sep_token)

        if not obj_sequence:
            continue

        seq_tensor = torch.tensor(np.vstack(obj_sequence), dtype=torch.float32)

        dataset.append({
            "id": obj_id,
            "sequence": seq_tensor,
            "type": city_obj.get("type", "Unknown")
        })

    return dataset

In [10]:
seq_dataset = create_sequence_dataset(data_path)